# Real-data NeuralProphet decomposition

Paper figures for **Beijing AQI** and **FSB campus load**.

This notebook does *not* compare to a known DGP (that is simulation Figure 3). It extracts the same NeuralProphet pieces — trend, daily seasonality, weekly seasonality, covariate contribution — from the two-stage model fit on each real series, and writes PDF/PNG files you can drop into `figures/`.

The energy section requires the full load record (`energy_data.csv`), which is not redistributed. See `data/README.md`. The Beijing section runs from the repository as-is.

Run in the `datamine_np` environment, with `OrdinalNeuralProphet.py` in the working directory (or on `PYTHONPATH`). Fits take several minutes per series (`epochs=50`).

Expected files next to this notebook, or set the paths in the config cell:
- `energy_data.csv`
- `BeijingAQI_data.csv`


In [ ]:
import os, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
logging.getLogger("NP").setLevel(logging.ERROR)

from OrdinalNeuralProphet import OrdinalNeuralProphet

OUTDIR = Path("figures")
OUTDIR.mkdir(exist_ok=True)

# Paper-style matplotlib defaults
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
})
print("output dir:", OUTDIR.resolve())


## Config

Edit paths if the CSVs are not in the working directory. Model settings match the horizon experiments (`n_lags=24`, `n_forecasts=12`, `H=72` holdout, `val_mode='refit'`).


In [ ]:
SERIES = {
    "beijing": dict(
        csv="BeijingAQI_data.csv",
        target="aqi_category",
        lagged=["TEMP", "DEWP", "PRES", "Iws"],
        future=["TEMP", "DEWP", "PRES", "Iws"],
        title="Beijing AQI",
        ar_reg=0.3,
    ),
    "energy": dict(
        csv="energy_data.csv",
        target="load_category",
        lagged=["temp"],
        future=["temp", "is_not_weekend", "in_session", "is_not_holiday", "is_not_summerbreak"],
        title="FSB campus load",
        ar_reg=0.3,
    ),
}

FIT_KW = dict(
    seed=22,
    n_lags=24,
    n_forecasts=12,
    test_periods=72,
    val_periods=500,
    num_categories=4,
    forecast_only=False,
    round_intermediate=False,
    freq="h",
    epochs=50,
    val_mode="refit",
)


## Helpers

Same extract path as the simulation / Beijing notebook cell:

`make_future_dataframe(..., n_historic_predictions=True)` then `predict()`.


In [ ]:
def load_frame(csv):
    df = pd.read_csv(csv)
    time_col = "datetime" if "datetime" in df.columns else "ds"
    df[time_col] = pd.to_datetime(df[time_col])
    return df.set_index(time_col)


def fit_ordinal(name):
    cfg = SERIES[name]
    if not Path(cfg["csv"]).exists():
        raise FileNotFoundError(f"{cfg['csv']} not found. Put it next to the notebook or edit SERIES['{name}']['csv'].")
    df = load_frame(cfg["csv"])
    print(f"{name}: {len(df)} rows, cols={list(df.columns)}")
    model = OrdinalNeuralProphet(
        df, cfg["target"],
        covariates=cfg["lagged"],
        future_covariates=cfg["future"],
        ar_reg=cfg["ar_reg"],
        **FIT_KW,
    )
    return model


def extract_components(model):
    m = model.fitted_model
    future = m.make_future_dataframe(df=model.prophet_data, n_historic_predictions=True, periods=0)
    fc = m.predict(future)
    keep = [c for c in [
        "ds", "y", "yhat1", "trend",
        "season_daily", "season_weekly", "season_yearly",
        "future_regressors_additive", "lagged_regressors_additive",
    ] if c in fc.columns]
    extra = [c for c in fc.columns if c.startswith("future_regressor") or c.startswith("lagged_regressor")]
    out = fc[keep + [c for c in extra if c not in keep]].copy()
    out = out.dropna(subset=[c for c in ["trend", "yhat1"] if c in out.columns]).reset_index(drop=True)
    out["ds"] = pd.to_datetime(out["ds"])
    return out


def amplitude_table(fc, name):
    def sd(col):
        return float(np.asarray(fc[col], dtype=float).std()) if col in fc.columns else np.nan
    struct_cols = [c for c in ["trend", "season_daily", "season_weekly", "future_regressors_additive"] if c in fc.columns]
    resid_sd = np.nan
    if "y" in fc.columns and "yhat1" in fc.columns:
        resid_sd = float((fc["y"].astype(float) - fc["yhat1"].astype(float)).std())
    structural_sd = float(fc[struct_cols].astype(float).sum(axis=1).std()) if struct_cols else np.nan
    row = dict(
        series=name,
        n=len(fc),
        daily_sd=sd("season_daily"),
        weekly_sd=sd("season_weekly"),
        covariate_sd=sd("future_regressors_additive"),
        trend_sd=sd("trend"),
        resid_sd=resid_sd,
        structural_sd=structural_sd,
        noise_to_signal=(resid_sd / structural_sd) if structural_sd else np.nan,
    )
    return row


def savefig(fig, stem):
    pdf = OUTDIR / f"{stem}.pdf"
    png = OUTDIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, bbox_inches="tight")
    print("wrote", pdf, "and", png)


## Figure helper (Figure 3 layout, no ground-truth overlay)


In [ ]:
COLOR = "#0072B2"
COLOR_COV = "#D55E00"


def plot_decomposition(fc, title, stem):
    fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.0))
    t = fc["ds"]

    ax = axes[0, 0]
    if "trend" in fc.columns:
        tr = fc["trend"].astype(float)
        ax.plot(t, tr - tr.mean(), color=COLOR, lw=1.1)
    ax.set_title("(a) Trend (centered)")
    ax.set_ylabel("Deviation from mean")
    for lbl in ax.get_xticklabels():          # autofmt_xdate() takes no ax= argument
        lbl.set_rotation(30)
        lbl.set_horizontalalignment("right")

    ax = axes[0, 1]
    if "season_daily" in fc.columns:
        daily = fc["season_daily"].astype(float)
        hour = fc["ds"].dt.hour
        prof = daily.groupby(hour).mean()
        ax.plot(prof.index, prof.values, color=COLOR, lw=1.8)
        ax.fill_between(prof.index, prof.values, alpha=0.12, color=COLOR)
        ax.set_xticks([0, 6, 12, 18, 23])
    ax.set_title("(b) Daily seasonality")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Component value")

    ax = axes[1, 0]
    if "season_weekly" in fc.columns:
        weekly = fc["season_weekly"].astype(float)
        dow = fc["ds"].dt.dayofweek
        prof = weekly.groupby(dow).mean()
        ax.plot(prof.index, prof.values, color=COLOR, lw=1.8, marker="o")
        ax.set_xticks(range(7))
        ax.set_xticklabels(list("MTWTFSS"))
    ax.set_title("(c) Weekly seasonality")
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Component value")

    ax = axes[1, 1]
    if "future_regressors_additive" in fc.columns:
        ax.plot(t, fc["future_regressors_additive"].astype(float),
                color=COLOR_COV, lw=0.55, alpha=0.85)
    ax.set_title("(d) Covariate contribution")
    ax.set_xlabel("t")
    ax.set_ylabel("Component value")

    fig.suptitle(title, y=1.02, fontsize=12)
    fig.tight_layout()
    savefig(fig, stem)
    return fig


def plot_daily_weekly_compare(store, stem="real_cycles_compare"):
    """Side-by-side daily / weekly profiles for both series. Good paper figure."""
    fig, axes = plt.subplots(2, 2, figsize=(10.6, 6.4), sharey=False)
    order = ["beijing", "energy"]
    for i, name in enumerate(order):
        if name not in store:
            continue
        fc = store[name]["fc"]
        title = SERIES[name]["title"]
        ax = axes[0, i]
        if "season_daily" in fc.columns:
            prof = fc["season_daily"].astype(float).groupby(fc["ds"].dt.hour).mean()
            ax.plot(prof.index, prof.values, color=COLOR, lw=1.8)
            ax.fill_between(prof.index, prof.values, alpha=0.12, color=COLOR)
        ax.set_title(f"{title}: daily")
        ax.set_xlabel("Hour of day")
        ax.set_xticks([0, 6, 12, 18, 23])
        if i == 0:
            ax.set_ylabel("Component value")

        ax = axes[1, i]
        if "season_weekly" in fc.columns:
            prof = fc["season_weekly"].astype(float).groupby(fc["ds"].dt.dayofweek).mean()
            ax.plot(prof.index, prof.values, color=COLOR, lw=1.8, marker="o")
        ax.set_title(f"{title}: weekly")
        ax.set_xlabel("Day of week")
        ax.set_xticks(range(7))
        ax.set_xticklabels(list("MTWTFSS"))
        if i == 0:
            ax.set_ylabel("Component value")
    fig.tight_layout()
    savefig(fig, stem)
    return fig


## Fit FSB energy and plot


In [ ]:
store = {}

model_e = fit_ordinal("energy")
fc_e = extract_components(model_e)
store["energy"] = dict(model=model_e, fc=fc_e)
print("energy component columns:", list(fc_e.columns))
print(pd.Series(amplitude_table(fc_e, "energy")))
fig_e = plot_decomposition(fc_e, "FSB campus load", "energy_decomposition")
fc_e.to_csv(OUTDIR / "energy_components.csv", index=False)


## Fit Beijing AQI and plot


In [ ]:
model_b = fit_ordinal("beijing")
fc_b = extract_components(model_b)
store["beijing"] = dict(model=model_b, fc=fc_b)
print("beijing component columns:", list(fc_b.columns))
print(pd.Series(amplitude_table(fc_b, "beijing")))
fig_b = plot_decomposition(fc_b, "Beijing AQI", "beijing_decomposition")
fc_b.to_csv(OUTDIR / "beijing_components.csv", index=False)


## Comparison table and joint cycle figure

Component SDs on the integer-label scale. These numbers go in the real-data interpretability paragraph.

You already measured Beijing in an earlier notebook: daily \(0.076\), weekly \(0.042\), covariates \(1.182\), NSR \(0.423\). Re-running here keeps both series on the same seed and code path.


In [ ]:
amp = pd.DataFrame([amplitude_table(store[n]["fc"], n) for n in store])
amp.to_csv(OUTDIR / "real_component_sd.csv", index=False)
display(amp.round(3))
fig_c = plot_daily_weekly_compare(store, "real_cycles_compare")


## Combined per-regressor contributions (paper figure)

One figure for both series, panels `(a)` and `(b)` rather than a small title over each.
Two points that matter for print quality:

* the figure is authored at roughly its final printed width (`figsize=(6.9, 4.5)`), so it is
  placed at `\\textwidth` at about 1:1 and the fonts are **not** shrunk;
* `fig.subfigures` lets the two columns hold different numbers of panels (4 vs 5) while
  still aligning top and bottom.

Regressors are ordered by the magnitude of their contribution, so the dominant term is on top.

> Requires matplotlib >= 3.4 for `Figure.subfigures`.


In [ ]:
COLOR_COV = "#D55E00"

PRETTY = {
    "DEWP": "Dew point", "TEMP": "Temperature", "PRES": "Pressure", "Iws": "Wind speed",
    "temp": "Temperature", "is_not_weekend": "Weekday", "in_session": "In session",
    "is_not_holiday": "Not holiday", "is_not_summerbreak": "In term",
}
# largest-contribution term first within each series
ORDER = {
    "beijing": ["DEWP", "PRES", "TEMP", "Iws"],
    "energy":  ["temp", "is_not_weekend", "is_not_holiday", "in_session", "is_not_summerbreak"],
}


def regressor_cols(fc):
    """Map a short covariate name -> its per-regressor column in the predict() frame."""
    out = {}
    for c in fc.columns:
        if c.startswith("future_regressor") and c != "future_regressors_additive":
            out[c.replace("future_regressor_", "").replace("__fut", "")] = c
    return out


def plot_regressors_combined(store, stem="real_regressors",
                             labels=("(a) Beijing AQI", "(b) FSB campus load")):
    """Both series in one figure, authored at final printed size."""
    missing = [n for n in ("beijing", "energy") if n not in store]
    if missing:
        raise KeyError(f"store is missing {missing}; run the fit cells for both series first")

    fig = plt.figure(figsize=(6.9, 4.5))
    subfigs = fig.subfigures(1, 2, wspace=0.10)

    for sf, name, lab in zip(subfigs, ["beijing", "energy"], labels):
        fc = store[name]["fc"]
        avail = regressor_cols(fc)
        if not avail:
            print(name, ": no per-regressor columns in predict() frame")
            continue
        keys = ([k for k in ORDER[name] if k in avail]
                + [k for k in avail if k not in ORDER[name]])

        axes = sf.subplots(len(keys), 1, sharex=True)
        axes = [axes] if len(keys) == 1 else list(axes)
        for ax, k in zip(axes, keys):
            ax.plot(fc["ds"], fc[avail[k]].astype(float), color=COLOR_COV, lw=0.55)
            ax.set_ylabel(PRETTY.get(k, k), fontsize=6.5)
            ax.tick_params(labelsize=5.8)
            ax.grid(True, alpha=0.25, lw=0.5)
            ax.margins(x=0.01)
            for s in ("top", "right"):
                ax.spines[s].set_visible(False)

        axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=5))
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        axes[-1].tick_params(axis="x", labelsize=6.5)
        sf.suptitle(lab, fontsize=9)
        sf.subplots_adjust(left=0.22, right=0.99, top=0.92, bottom=0.08, hspace=0.25)

    savefig(fig, stem)
    return fig


plot_regressors_combined(store)


## Files written

| File | Use |
|---|---|
| `figures/energy_decomposition.pdf` | FSB analog of Figure 3 |
| `figures/beijing_decomposition.pdf` | Beijing analog of Figure 3 |
| `figures/real_cycles_compare.pdf` | Side-by-side daily/weekly (paper figure) |
| `figures/real_regressors.pdf` | Combined per-regressor contributions, panels (a)/(b) (paper figure) |
| `figures/real_component_sd.csv` | SDs for the text |
| `figures/*_components.csv` | Full extracted frames |

After this runs, send `real_component_sd.csv` and the three PDFs and we will drop them into `Main.tex`.
